# US Urban Migration Patterns (2000-2020)**Author:** Najiba — Oregon State University  **Date:** June 2026This notebook analyzes population change in U.S. cities between 2000 and 2020, focusing on the top 10 growing and top 10 shrinking cities. It uses the **Global Urban Demographic Dataset (GUDD)** from Zimmer et al. (2026, Nature Cities).## Data Source> Zimmer, A., Brooks, N., Gaughan, A. E., Tuholske, C. (2026). > *Global Divergence in Urban Demographic Change and Migration Patterns.* > Nature Cities. > Original repository: https://github.com/zimmermaps/urban_demographyThe visualization style and color palettes are adapted from the original Zimmer et al. notebooks `07_fun_figures.ipynb` and `09_figure4a_migration_map.ipynb`, with attribution.

## 1. Setup and Data Loading

In [ ]:
import osimport pandas as pdimport numpy as npimport matplotlib as mplimport matplotlib.colors as mcolorsimport matplotlib.pyplot as pltimport geopandas as gpdimport seaborn as snsfrom matplotlib.font_manager import FontPropertiesfrom matplotlib.animation import FuncAnimation, PillowWriterfrom matplotlib.ticker import FuncFormatterfrom matplotlib.patches import Patchimport foliumfrom branca.colormap import LinearColormapfrom IPython.display import HTML# Style (from Zimmer et al. paper)mpl.rcParams['pdf.fonttype'] = 42mpl.rcParams['font.family'] = 'DejaVu Sans Mono'mpl.rcParams['font.size'] = 10sns.set(style='whitegrid')mono_font = FontProperties(family='DejaVu Sans Mono', size=10)

In [ ]:
# === DATA PATHS ===# Update this path to where you downloaded the GUDD dataDATA_FOLDER = './data'# Load annual metrics (city x year demographic indicators)df_annual = pd.read_csv(os.path.join(DATA_FOLDER, 'gudd_annual_metrics_static_boundaries.csv'))print(f'Annual metrics loaded: {df_annual.shape}')# Load change data (2000-2020 deltas)df_change = pd.read_csv(os.path.join(DATA_FOLDER, 'gudd_change_2000_2020_static_boundaries.csv'))print(f'Change data loaded: {df_change.shape}')# Load world boundaries for mapsworld = gpd.read_file(os.path.join(DATA_FOLDER, 'mapping', 'ne_50m_admin_0_countries.shp'))print(f'World boundaries loaded: {world.shape}')

## 2. Define Focus CitiesWe focus on the 10 fastest-growing and 10 fastest-shrinking U.S. cities between 2000 and 2020.

In [ ]:
growing_cities = ['Los Angeles', 'Las Vegas', 'Miami', 'Phoenix', 'Dallas',                   'Mesa', 'Washington', 'Orlando', 'Sacramento', 'Portland']shrinking_cities = ['Chicago', 'Detroit', 'New Orleans', 'Cleveland', 'St. Louis',                    'Pittsburgh', 'Buffalo', 'Cincinnati', 'Dayton', 'Toledo']focus_cities = growing_cities + shrinking_cities# Filter US data onlyus_change = df_change[df_change['Country'] == 'United States'].dropna(    subset=['longitude', 'latitude', 'total_pop_Delta', 'perc_from_migration']).copy()top_growing = us_change.nlargest(10, 'total_pop_Delta')top_shrinking = us_change.nsmallest(10, 'total_pop_Delta')print('TOP 10 GROWING US CITIES (2000-2020)')print(top_growing[['Name', 'total_pop_Delta', 'perc_from_migration']].to_string(index=False))print()print('TOP 10 SHRINKING US CITIES (2000-2020)')print(top_shrinking[['Name', 'total_pop_Delta']].to_string(index=False))

## 3. Interactive Map — Top 10 Growing and Shrinking US CitiesUses Folium with the Zimmer et al. color palette.

In [ ]:
# Zimmer color palette (from the original paper)migration_colors = [    '#fdd070', '#fdae61', '#f98e52', '#f46d43', '#e34a33', '#d73027',    '#c51b7d', '#ae017e', '#8c0273', '#5f0165', '#2b0040']colormap = LinearColormap(migration_colors, vmin=0, vmax=100,                          caption='% growth from migration')m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')# Growing cities (colored by migration percentage)max_growth = top_growing['total_pop_Delta'].max()for _, row in top_growing.iterrows():    radius = 10 + (row['total_pop_Delta'] / max_growth) * 25    folium.CircleMarker(        location=[row['latitude'], row['longitude']],        radius=radius,        popup=folium.Popup(            f"""<b>{row['Name']}</b><br>            Growth: +{row['total_pop_Delta']:,.0f}<br>            From migration: {row['perc_from_migration']:.1f}%<br>            From births: {100 - row['perc_from_migration']:.1f}%""",            max_width=300        ),        tooltip=f"{row['Name']}: +{row['total_pop_Delta']:,.0f}",        color=colormap(min(100, max(0, row['perc_from_migration']))),        fill=True,        fillColor=colormap(min(100, max(0, row['perc_from_migration']))),        fillOpacity=0.8, weight=2    ).add_to(m)# Shrinking citiesmax_loss = abs(top_shrinking['total_pop_Delta'].min())for _, row in top_shrinking.iterrows():    radius = 10 + (abs(row['total_pop_Delta']) / max_loss) * 25    folium.CircleMarker(        location=[row['latitude'], row['longitude']],        radius=radius,        popup=folium.Popup(            f"""<b>{row['Name']}</b><br>            Loss: {row['total_pop_Delta']:,.0f}<br>            2020 Pop: {row['total_pop_2020']:,.0f}""",            max_width=300        ),        tooltip=f"{row['Name']}: {row['total_pop_Delta']:,.0f}",        color='darkred', fill=True, fillColor='#8B0000',        fillOpacity=0.8, weight=2    ).add_to(m)colormap.add_to(m)m

## 4. Animated Plot — Population vs MigrationEach city's trajectory over 2000-2020. Green = growing, Red = shrinking.

In [ ]:
# Prepare focus city datadf_focus = df_annual[    (df_annual['Country'] == 'United States') &    (df_annual['Name'].isin(focus_cities))].copy()df_focus = df_focus[df_focus['total_pop'] > 0].sort_values('year').dropna(    subset=['total_pop', 'migration', 'births'])years = sorted(df_focus['year'].unique())# Assign colors: green shades for growing, red shades for shrinkinggreen_palette = plt.cm.Greens(np.linspace(0.4, 0.95, 10))red_palette = plt.cm.Reds(np.linspace(0.4, 0.95, 10))city_color_map = {}for i, city in enumerate(growing_cities):    city_color_map[city] = green_palette[i]for i, city in enumerate(shrinking_cities):    city_color_map[city] = red_palette[i]name_to_id = df_focus.drop_duplicates('ID_UC_G0').set_index('Name')['ID_UC_G0'].to_dict()locations = sorted(df_focus['ID_UC_G0'].unique())color_map = {}for city, color in city_color_map.items():    if city in name_to_id:        color_map[name_to_id[city]] = colordef human_format(x, pos):    if abs(x) >= 1e9: return f'{int(x/1e9)}B'    if abs(x) >= 1e6: return f'{int(x/1e6)}M'    if abs(x) >= 1e3: return f'{int(x/1e3)}K'    return str(int(x))formatter = FuncFormatter(human_format)

In [ ]:
# Animated: Population vs Migrationpop_min = df_focus['total_pop'].min()pop_max = df_focus['total_pop'].max()mig_min = df_focus['migration'].min()mig_max = df_focus['migration'].max()fig, ax = plt.subplots(figsize=(12, 7))plt.close(fig)def update_migration(frame):    year = years[frame]    ax.clear()    past = df_focus[df_focus['year'] <= year]    current = df_focus[df_focus['year'] == year]        ax.set_xscale('log')    ax.set_xlim(pop_min * 0.9, pop_max * 1.1)    ax.set_ylim(mig_min * 1.1, mig_max * 1.1)    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)        ax.set_xlabel('Total Population', fontproperties=mono_font, fontsize=12)    ax.set_ylabel('Net Migration (per year)', fontproperties=mono_font, fontsize=12)    ax.set_title(f'US Cities — Population vs Migration | {year}', fontsize=14)    ax.xaxis.set_major_formatter(formatter)    ax.yaxis.set_major_formatter(formatter)    ax.grid(True, color='gray', linestyle='--', linewidth=0.5, alpha=0.3)        for loc in locations:        if loc not in color_map: continue        loc_data = past[past['ID_UC_G0'] == loc]        if len(loc_data) > 1:            trail = loc_data.iloc[:-1]            ax.scatter(trail['total_pop'], trail['migration'],                      s=12, color=color_map[loc], alpha=0.25,                      edgecolor='black', linewidth=0.3)        for loc in locations:        if loc not in color_map: continue        loc_data = current[current['ID_UC_G0'] == loc]        if len(loc_data) == 0: continue        ax.scatter(loc_data['total_pop'], loc_data['migration'],                  s=100, color=color_map[loc],                  edgecolor='black', linewidth=0.8)        for _, row in loc_data.iterrows():            ax.annotate(row['Name'],                       xy=(row['total_pop'], row['migration']),                       xytext=(7, 5), textcoords='offset points',                       fontsize=8, fontweight='bold')        ax.legend(handles=[        Patch(facecolor='#2ca02c', label='Growing (10 cities)'),        Patch(facecolor='#d62728', label='Shrinking (10 cities)')    ], loc='lower left', fontsize=10)anim_mig = FuncAnimation(fig, update_migration, frames=len(years), interval=500, repeat=True)anim_mig.save('figures/us_focus_migration.gif', writer=PillowWriter(fps=2))print('Saved: figures/us_focus_migration.gif')HTML(anim_mig.to_jshtml())

## 5. Animated Plot — Population vs Births

In [ ]:
births_max = df_focus['births'].max()fig2, ax2 = plt.subplots(figsize=(12, 7))plt.close(fig2)def update_births(frame):    year = years[frame]    ax2.clear()    past = df_focus[df_focus['year'] <= year]    current = df_focus[df_focus['year'] == year]        ax2.set_xscale('log')    ax2.set_xlim(pop_min * 0.9, pop_max * 1.1)    ax2.set_ylim(0, births_max * 1.1)        ax2.set_xlabel('Total Population', fontproperties=mono_font, fontsize=12)    ax2.set_ylabel('Annual Births', fontproperties=mono_font, fontsize=12)    ax2.set_title(f'US Cities — Population vs Births | {year}', fontsize=14)    ax2.xaxis.set_major_formatter(formatter)    ax2.yaxis.set_major_formatter(formatter)    ax2.grid(True, color='gray', linestyle='--', linewidth=0.5, alpha=0.3)        for loc in locations:        if loc not in color_map: continue        loc_data = past[past['ID_UC_G0'] == loc]        if len(loc_data) > 1:            trail = loc_data.iloc[:-1]            ax2.scatter(trail['total_pop'], trail['births'],                       s=12, color=color_map[loc], alpha=0.25,                       edgecolor='black', linewidth=0.3)        for loc in locations:        if loc not in color_map: continue        loc_data = current[current['ID_UC_G0'] == loc]        if len(loc_data) == 0: continue        ax2.scatter(loc_data['total_pop'], loc_data['births'],                   s=100, color=color_map[loc],                   edgecolor='black', linewidth=0.8)        for _, row in loc_data.iterrows():            ax2.annotate(row['Name'],                        xy=(row['total_pop'], row['births']),                        xytext=(7, 5), textcoords='offset points',                        fontsize=8, fontweight='bold')        ax2.legend(handles=[        Patch(facecolor='#2ca02c', label='Growing (10 cities)'),        Patch(facecolor='#d62728', label='Shrinking (10 cities)')    ], loc='upper left', fontsize=10)anim_births = FuncAnimation(fig2, update_births, frames=len(years), interval=500, repeat=True)anim_births.save('figures/us_focus_births.gif', writer=PillowWriter(fps=2))print('Saved: figures/us_focus_births.gif')HTML(anim_births.to_jshtml())

## 6. Key Findings| Pattern | Insight ||---|---|| **Sunbelt Boom** | Phoenix, Las Vegas, Miami grew primarily through migration || **Rust Belt Decline** | Detroit, Chicago, Cleveland lost hundreds of thousands || **New Orleans Anomaly** | Shrank due to Hurricane Katrina (2005), not industrial decline || **Births Trend** | Generally declined across most cities, more pronounced in shrinking ones |## AcknowledgmentsThis analysis would not be possible without the work of Zimmer et al. (2026) and the open release of the GUDD dataset. Visualization style and color palettes are adapted from the original repository with full attribution.## LicenseMIT License — see [LICENSE](../LICENSE)